# 05 — Исследование на году 1

**Цель:** улучшить стратегию v1, глядя **только на год 1** (окт 2024 – сен 2025).
Год 2 не трогаем до самой последней проверки. Иначе мы бы подгоняли стратегию под данные, на которых потом её оцениваем.

**Правила исследования:**
- меняем **одну вещь за раз**, тогда понятно, что именно помогло;
- каждый запуск пишем в журнал `log`, а в отчёте честно показываем, сколько вариантов попробовали;
- смотрим два уровня комиссий: **VIP-7** (дешёвый) и **Regular** (дорогой). VIP-4 всегда между ними;
- выбираем **плато** (несколько соседних значений работают похоже хорошо), а не одну случайно лучшую точку.

## 1. Setup

Ноутбук лежит в папке `notebooks/`, а наш код — в корне проекта (`config.py`, `src/`).
Поэтому добавляем корень проекта в `sys.path`, иначе `import config` не найдётся.

In [4]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent                 # notebooks/ → на папку выше = корень проекта
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))        # теперь Python видит config.py и src/

from datetime import datetime, timezone
import numpy as np
import polars as pl

import config
from src.funding import load_funding, funding_on_bars
from src.research import evaluate        # один прогон стратегии → метрики

## 2. Данные и фандинг

- `df` — 5-минутные свечи пары. Колонки с `_a` относятся к ETH, с `_b` к BTC.
- `fa`, `fb` — ставка фандинга **на каждой свече**. В свечи 00:00, 08:00 и 16:00 UTC это настоящая ставка, во всех остальных 0.

In [5]:
df = pl.read_parquet(config.PROCESSED_DIR / "pair_5m.parquet")

open_time = df["open_time"].to_numpy()
fa = funding_on_bars(open_time, load_funding(config.SYMBOL_A))     # фандинг ETH на каждой свече
fb = funding_on_bars(open_time, load_funding(config.SYMBOL_B))     # фандинг BTC на каждой свече

print("свечей:", df.shape[0], "| фандинг:", fa.shape, fb.shape)   # все три длины должны совпасть

свечей: 210240 | фандинг: (210240,) (210240,)


## 3. Граница года 1

`split` — номер первой свечи года 2 (2025-10-01 00:00 UTC).

- Год 1 = свечи с `0` до `split` (не включая).
- В году 365 дней × 288 свечей в день, поэтому ожидаем `split = 105 120`.

In [6]:
oos = datetime.fromisoformat(config.OOS_START).replace(tzinfo=timezone.utc)   # "2025-10-01" → дата в UTC
oos_ms = int(oos.timestamp() * 1000)                                         # → миллисекунды, как open_time

split = int((df["open_time"] < oos_ms).sum())    # сколько свечей раньше начала года 2
print("split =", split)
assert split == 105_120                          # если не так — с данными что-то не то

split = 105120


## 4. Журнал экспериментов

`record(name, **params)` делает три вещи:
1. прогоняет стратегию на году 1 **два раза**, для VIP-7 и для Regular;
2. дописывает обе строки в журнал `log`;
3. возвращает весь журнал как таблицу.

`params` — то, что меняем в этом эксперименте, например `max_loss=200.0`.
Всё, что не передали, берётся из `config` (окно 288, вход 2, стоп 4, время 576).

Пример: `record("loss $200", max_loss=200.0)` → в журнале появятся 2 строки: vip7 и regular.

In [7]:
log = []                                  # сюда копятся ВСЕ запуски
TIERS = ("vip7", "regular")               # дешёвые и дорогие комиссии

# какие колонки показывать (порядок = порядок в таблице)
COLS = ["name", "tier",
        "window", "entry_z", "stop_z", "max_hold", "max_loss", "max_loss_pct",   # параметры
        "n_trades", "win_rate", "avg_trade", "profit_factor",                    # сделки
        "total_return", "max_drawdown", "sharpe", "exposure"]                    # капитал


def record(name, **params):
    for tier in TIERS:
        row = evaluate(df, fa, fb, split, tier=tier, **params)   # один прогон → словарь метрик
        row["name"] = name                                       # подпись эксперимента
        log.append(row)
    table = pl.DataFrame(log)
    return table.select([c for c in COLS if c in table.columns])   # показываем только существующие колонки

## 5. Baseline — v1 на году 1

Оба новых стопа по умолчанию выключены (`0`), поэтому результат **должен совпасть** со старым.
Это проверка, что новый код ничего не сломал.

| VIP-7         | ожидаем |
|---------------|---------|
| сделок        | 483     |
| return        | −34.7%  |
| max drawdown  | −37.2%  |
| profit factor | 0.768   |
| Sharpe        | −1.76   |

In [8]:
table = record("baseline")
table

name,tier,window,entry_z,stop_z,max_hold,max_loss,max_loss_pct,n_trades,win_rate,avg_trade,profit_factor,total_return,max_drawdown,sharpe,exposure
str,str,i64,f64,f64,i64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64
"""baseline""","""vip7""",288,2.0,4.0,576,0.0,0.0,483,0.616977,-14.350058,0.767516,-0.346554,-0.372496,-1.758088,0.558381
"""baseline""","""regular""",288,2.0,4.0,576,0.0,0.0,483,0.581781,-25.559567,0.613962,-0.617264,-0.631405,-3.172942,0.558381


## 6. Эксперимент 4.1 — стоп по убытку сделки

**Проблема v1.** Когда спред трендит, скользящее среднее едет вслед за ним, и z остаётся около 2–3.
z-стоп (|z| > 4) не срабатывает, сделка висит до time-стопа (2 дня) и теряет до **−$1,920**.

**Новое правило.** Если открытая сделка в минусе больше порога, выходим на open следующей свечи
(причина `loss`), потом пауза, пока |z| не станет меньше 2.

**Как сработает на примере** (шорт спреда, порог $200):
- вход: ETH = 2,500 (шорт 4 ETH), BTC = 62,500 (лонг 0.16 BTC), комиссия −$6.40;
- ETH вырос до 2,550, BTC стоит: −4 × 50 − 6.40 = **−206.40** < −200 → выходим.
- Проще говоря, $200 ≈ ETH обогнал BTC на ~2%.

**Два способа задать порог:**

| | Вариант 1: % от ноги | Вариант 2: % от капитала при входе |
|---|---|---|
| Формула | % × $10,000 | % × капитал перед входом |
| Капитал $20,000, 1% / 2% | $200 | $200 |
| Капитал $13,000 | $200 (не меняется) | $130 (строже) |

Сетки подобраны так, что при капитале $20,000 оба варианта дают одинаковые $100 … $500.
Разница появляется только когда капитал падает.

In [9]:
# Вариант 1: % от ноги ($10,000) → $100, $150, $200, $300, $500
for pct in (0.01, 0.015, 0.02, 0.03, 0.05):
    table = record(f"loss {pct:.1%} of leg", max_loss=pct * config.NOTIONAL)

# Вариант 2: % от капитала при входе (при $20,000 → те же $100 … $500)
for pct in (0.005, 0.0075, 0.01, 0.015, 0.025):
    table = record(f"loss {pct:.2%} of equity", max_loss_pct=pct)

table

name,tier,window,entry_z,stop_z,max_hold,max_loss,max_loss_pct,n_trades,win_rate,avg_trade,profit_factor,total_return,max_drawdown,sharpe,exposure
str,str,i64,f64,f64,i64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64
"""baseline""","""vip7""",288,2.0,4.0,576,0.0,0.0,483,0.616977,-14.350058,0.767516,-0.346554,-0.372496,-1.758088,0.558381
"""baseline""","""regular""",288,2.0,4.0,576,0.0,0.0,483,0.581781,-25.559567,0.613962,-0.617264,-0.631405,-3.172942,0.558381
"""loss 1.0% of leg""","""vip7""",288,2.0,4.0,576,100.0,0.0,641,0.513261,-10.65475,0.811099,-0.341485,-0.358297,-2.00857,0.445234
"""loss 1.0% of leg""","""regular""",288,2.0,4.0,576,100.0,0.0,651,0.480799,-23.878692,0.616333,-0.777251,-0.788833,-4.24403,0.441486
"""loss 1.5% of leg""","""vip7""",288,2.0,4.0,576,150.0,0.0,579,0.571675,-13.050018,0.78678,-0.377798,-0.391538,-2.185896,0.484551
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""loss 1.00% of equity""","""regular""",288,2.0,4.0,576,0.0,0.01,613,0.512235,-22.566038,0.643651,-0.691649,-0.698958,-4.068651,0.462224
"""loss 1.50% of equity""","""vip7""",288,2.0,4.0,576,0.0,0.015,536,0.608209,-15.310086,0.762869,-0.41031,-0.434928,-2.173274,0.516134
"""loss 1.50% of equity""","""regular""",288,2.0,4.0,576,0.0,0.015,563,0.557726,-24.423556,0.62934,-0.687523,-0.699843,-3.778743,0.495786


In [10]:
with pl.Config(tbl_rows=-1):
    print(table.filter(pl.col("tier") == "vip7")
               .select("name", "n_trades", "win_rate", "avg_trade",
                       "profit_factor", "total_return", "max_drawdown", "sharpe"))

shape: (11, 8)
┌────────────┬──────────┬──────────┬────────────┬────────────┬────────────┬────────────┬───────────┐
│ name       ┆ n_trades ┆ win_rate ┆ avg_trade  ┆ profit_fac ┆ total_retu ┆ max_drawdo ┆ sharpe    │
│ ---        ┆ ---      ┆ ---      ┆ ---        ┆ tor        ┆ rn         ┆ wn         ┆ ---       │
│ str        ┆ i64      ┆ f64      ┆ f64        ┆ ---        ┆ ---        ┆ ---        ┆ f64       │
│            ┆          ┆          ┆            ┆ f64        ┆ f64        ┆ f64        ┆           │
╞════════════╪══════════╪══════════╪════════════╪════════════╪════════════╪════════════╪═══════════╡
│ baseline   ┆ 483      ┆ 0.616977 ┆ -14.350058 ┆ 0.767516   ┆ -0.346554  ┆ -0.372496  ┆ -1.758088 │
│ loss 1.0%  ┆ 641      ┆ 0.513261 ┆ -10.65475  ┆ 0.811099   ┆ -0.341485  ┆ -0.358297  ┆ -2.00857  │
│ of leg     ┆          ┆          ┆            ┆            ┆            ┆            ┆           │
│ loss 1.5%  ┆ 579      ┆ 0.571675 ┆ -13.050018 ┆ 0.78678    ┆ -0.377798  ┆ 

## 9. Вывод по эксперименту 4.1 — стоп по убытку сделки

### Что проверяли

**Проблема v1.** Когда спред трендит, скользящее среднее едет вслед за ним, и z остаётся около 2–3.
z-стоп (|z| > 4) не срабатывает, и сделка держится до time-стопа. Отсюда самые большие убытки (до −$1,920).

**Идея.** Выходить, если открытая сделка в минусе больше порога. Выход на open следующей свечи,
причина `loss`, потом пауза до |z| < 2.

**Варианты (10 штук, только год 1):**
- **% от ноги ($10,000):** 1%, 1.5%, 2%, 3%, 5% → порог $100 … $500;
- **% от капитала при входе:** 0.5%, 0.75%, 1%, 1.5%, 2.5% → при $20,000 те же $100 … $500, при падении капитала строже.

### Результаты (VIP-7)

| Вариант          | Сделок | Win rate | Avg trade | PF    | Return     | Max DD     | Sharpe    |
|------------------|--------|----------|-----------|-------|------------|------------|-----------|
| **baseline**     | 483    | 61.7%    | −14.35    | 0.768 | **−34.7%** | **−37.2%** | **−1.76** |
| 1% ноги          | 641    | 51.3%    | −10.65    | 0.811 | −34.1%     | −35.8%     | −2.01     |
| 1.5% ноги        | 579    | 57.2%    | −13.05    | 0.787 | −37.8%     | −39.2%     | −2.19     |
| 2% ноги          | 546    | 59.9%    | −14.55    | 0.771 | −39.7%     | −42.4%     | −2.18     |
| 3% ноги          | 516    | 60.9%    | −17.87    | 0.731 | −46.1%     | −48.5%     | −2.42     |
| 5% ноги          | 496    | 61.5%    | −15.38    | 0.758 | −38.1%     | −40.7%     | −1.96     |
| 0.5% капитала    | 714    | 43.1%    | −13.64    | 0.746 | −48.7%     | −49.8%     | −3.03     |
| 0.75% капитала   | 616    | 53.4%    | −12.34    | 0.792 | −38.0%     | −39.7%     | −2.10     |
| 1% капитала      | 574    | 58.2%    | −13.42    | 0.782 | −38.5%     | −40.8%     | −2.23     |
| 1.5% капитала    | 536    | 60.8%    | −15.31    | 0.763 | −41.0%     | −43.5%     | −2.17     |
| 2.5% капитала    | 498    | 61.4%    | −14.72    | 0.765 | −36.7%     | −39.3%     | −1.92     |

На **Regular** проверенные варианты тоже хуже baseline: −69% … −78% против −61.7%.

### Что видно

1. **Sharpe хуже baseline во всех 10 вариантах.** Ни один порог не улучшил стратегию.
2. **Плато нет.** Результаты скачут без системы: 3% ноги хуже, чем 2% и 5%. Это шум, а не закономерность.
3. **Единственный «почти равный» вариант (1% ноги, −34.1%)** не стабилен: Sharpe хуже, а на Regular он падает до −77.7%.
4. **Даже мягкий стоп ($500) вредит.** Сделок всего на 13 больше, а return хуже на 3.5 п.п. (≈ $700).
5. **% от капитала не лучше, чем % от ноги.**

### Почему стоп не помогает

- **Он режет сделки, которые потом вернулись бы к среднему.**
  Многие сделки временно уходят в −$100…−$500, а затем закрываются по `mean` в плюс.
  Стоп превращает временный минус в настоящий убыток, поэтому win rate падает (61.7% → 43–61%).
- **Больше сделок — больше комиссий.**
  Пример: 1% ноги даёт 641 сделку вместо 483, то есть +158 × $12.80 ≈ **−$2,000** только на комиссиях (VIP-7).
  На Regular каждая сделка стоит $24, поэтому там ещё хуже.
- **Катастрофических сделок мало**, а стоп задевает сотни обычных.

### Решение

> **Стоп по убытку отклонён.** В v2 он выключен (`max_loss = 0`, `max_loss_pct = 0`).
> Код оставлен в движке и покрыт тестами, его можно включить в любой момент.

**Главный урок:** проблему тренда нельзя решить **на выходе**, потому что это режет и хорошие сделки.
Её надо решать **на входе**: не открывать сделку, пока спред трендит.
→ Следующий эксперимент: **4.4 — режимный фильтр**.

**Сколько вариантов попробовано:** 10 (+ baseline). Все записаны в `reports/research_log_year1.csv`.

In [11]:
out = ROOT / "reports" / "research_log_year1.csv"
pl.DataFrame(log).write_csv(out)
print("сохранено:", out, "| запусков в журнале:", len(log))

сохранено: /Users/system/Downloads/Quant_Project/reports/research_log_year1.csv | запусков в журнале: 22
